# Train and Predict Drones

### Abstract

...


### Introduction

...


### Dataset



# 🧠 1. Setup

pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml) and check software and hardware.

[![PyPI - Version](https://img.shields.io/pypi/v/ultralytics?logo=pypi&logoColor=white)](https://pypi.org/project/ultralytics/) [![PyPI - Python Version](https://img.shields.io/pypi/pyversions/ultralytics?logo=python&logoColor=gold)](https://pypi.org/project/ultralytics/)

### 1.0 Installing depedencies

In [ ]:
!pip install ultralytics markdown rich wrapt pandas mlflow huggingface_hub opencv-python wandb datasets -q

### 1.1 Importing Libraries

In [ ]:
from datasets import load_dataset, Image, concatenate_datasets, DatasetDict
from IPython.display import display, Image as IPyImage
from ultralytics import YOLO, settings
from huggingface_hub import login
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from datetime import datetime
from PIL import Image

import pandas as pd
import ultralytics
import numpy as np
import shutil
import wandb
import math
import re
import os

ultralytics.checks()

### 1.2 Global Definitions

In [ ]:
DATASET_ROOT_DIR = './datasets/main/'  # must end with /
DATASET_ALL_DIR = DATASET_ROOT_DIR + 'train_validation_test'
TRAINING_DATASET_DIRECTORY = DATASET_ROOT_DIR + 'train/'
VALIDATION_DATASET_DIRECTORY = DATASET_ROOT_DIR + 'valid/'
TEST_DATASET_DIRECTORY = DATASET_ROOT_DIR + 'test/'
MODELS_DIRECTORY = './models/'
RUNS_DIRECTORY = "./runs"
load_dotenv()

### 1.3 Global Settings

In [ ]:
# YOLO settings
settings.update({"wandb": True})

# Initialize Weights & Biases environment
wandb.login(key=os.getenv("WANDB_TOKEN"))

# login("TOKEN") # Keep commented if token loaded from .env file

### 1.4 Global Structure

In [ ]:
dataset_structure = {
    "path": "",
    "name": "",
    "classes": {
        0: "drone",
        1: "other"
    },
    "images": {
        "train": [],
        "val": [],
        "test": []
    },
    "labels": {
        "train": [],
        "val": [],
        "test": []
    }
}

### 1.5 Global Function Definitions

In [ ]:
def list_files(directory: str, extensions: list, include_root_directory=False, recursive=False):
    matched_files = []

    # Ensure extensions is a tuple for endswith
    ext_tuple = tuple(extensions) if isinstance(extensions, list) else extensions

    if recursive:
        for root, _, files in os.walk(directory):
            for f in files:
                if f.endswith(ext_tuple):
                    matched_files.append(os.path.join(root, f) if include_root_directory else f)
    else:
        for f in os.listdir(directory):
            if f.endswith(ext_tuple) and os.path.isfile(os.path.join(directory, f)):
                matched_files.append(os.path.join(directory, f) if include_root_directory else f)

    return matched_files


def numeric_key(name):
    """Extract the first number from a filename for sorting."""
    nums = re.findall(r'\d+', name)
    return int(nums[0]) if nums else float('inf')


def sort_files_by_number(files: list):
    """
    Sort a list of filenames by the first number found in each name.

    Args:
        files (list): List of filenames (strings)

    Returns:
        list: Sorted list of filenames
    """
    return sorted(files, key=numeric_key)


def update_dataset_structure():
    dataset_structure["path"] = DATASET_ROOT_DIR
    dataset_structure["name"] = DATASET_ROOT_DIR.split("/")[-2]

    dataset_structure["images"]["train"] = sort_files_by_number(
        list_files(TRAINING_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True)
    )
    dataset_structure["labels"]["train"] = sort_files_by_number(
        list_files(TRAINING_DATASET_DIRECTORY, [".txt"], True)
    )
    dataset_structure["images"]["valid"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True)
    )
    dataset_structure["labels"]["valid"] = sort_files_by_number(
        list_files(VALIDATION_DATASET_DIRECTORY, [".txt"], True)
    )
    dataset_structure["images"]["test"] = sort_files_by_number(
        list_files(TEST_DATASET_DIRECTORY, [".jpg", ".jpeg", ".JPG", ".JPEG"], True)
    )
    dataset_structure["labels"]["test"] = sort_files_by_number(
        list_files(TEST_DATASET_DIRECTORY, [".txt"], True)
    )
def delete_files_in_dataset(files_to_delete: list):
    try:
        confirm = input("Files are going to be deleted. Type 'yes' to continue: ").strip().lower()
        if confirm != 'yes':
            print("Deletion aborted by user.")
            return

        for file in files_to_delete:
            if os.path.isfile(file):
                os.remove(file)
                print(f"Deleted: {file}")
            else:
                print(f"Warning: File does not exist: {file}")

    except KeyboardInterrupt:
        print("\nDeletion aborted by user (KeyboardInterrupt).")
    finally:
        try:
            update_dataset_structure()
        except NameError:
            pass


def backup_dataset():
    dataset_path = dataset_structure.get("path", "")
    backup_dir = os.path.join(dataset_path, "backup")
    os.makedirs(backup_dir, exist_ok=True)

    now = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    target_name = f"{dataset_structure["name"]}-{now}"
    target_path = os.path.join(backup_dir, target_name)

    # Copytree with ignore to exclude the backup folder itself
    def ignore_backup(src, names):
        return {"backup"} if "backup" in names else set()

    shutil.copytree(dataset_path, target_path, ignore=ignore_backup)
    print(f"Backup created at: {target_path}")
    return str(target_path)

# 📂 2. Dataset

### 2.1 Acquire Dataset

Download dataset from hugging face.

In [ ]:
dataset = load_dataset("Hibou-Foundation/computer-vision")

In [ ]:
base_split = "train_validation_test"
label_column = "class_id"

train_ratio = [0.8, 0]   # [class 0, class 1]
valid_ratio = [0.2, 0]
test_ratio  = [0, 1]

seed = 42

for i in range(len(train_ratio)):
    assert train_ratio[i] + valid_ratio[i] + test_ratio[i] == 1.0

train_parts = []
valid_parts = []
test_parts = []

num_classes = len(train_ratio)

for cls in range(num_classes):
    cls_ds = dataset[base_split].filter(
        lambda x: x[label_column] == cls
    )

    cls_ds = cls_ds.shuffle(seed=seed)

    n = len(cls_ds)
    n_train = int(n * train_ratio[cls])
    n_valid = int(n * valid_ratio[cls])

    train_parts.append(cls_ds.select(range(0, n_train)))
    valid_parts.append(cls_ds.select(range(n_train, n_train + n_valid)))
    test_parts.append(cls_ds.select(range(n_train + n_valid, n)))

train_ds = concatenate_datasets(train_parts).shuffle(seed=seed)
valid_ds = concatenate_datasets(valid_parts).shuffle(seed=seed)
test_ds  = concatenate_datasets(test_parts).shuffle(seed=seed)

dataset = DatasetDict({
    "train": train_ds,
    "validation": valid_ds,
    "test": test_ds,
})
dataset

### 2.2 Convert to Yolo

In [ ]:
# Create folders
for split in ["train", "valid", "test"]:
    os.makedirs(f"{DATASET_ROOT_DIR}/{split}", exist_ok=True)


def export_to_yolo(ds, split_name):
    for idx, sample in enumerate(ds):
        image = sample["image"]  # already a PIL.Image
        label = sample["raw_label"]  # already YOLO format [[class, cx, cy, w, h], ...]
        img_name = sample["name"]
        txt_name = img_name.split(".")[0] + ".txt"

        # Save image
        img_path = f"{DATASET_ROOT_DIR}/{split_name}/{img_name}"
        image.save(img_path, quality=95)

        # Save labels
        lbl_path = f"{DATASET_ROOT_DIR}/{split_name}/{txt_name}"
        with open(lbl_path, "w") as f:
            f.write(label)


# Run export
split_mapping = {"train": "train", "validation": "valid", "test": "test"}
for hf_split, folder_name in split_mapping.items():
    export_to_yolo(dataset[hf_split], folder_name)

# ⚙️ 3. Training

Purpose: Handle model setup and training configuration.

### 3.1 Model Selection

In [ ]:
selected_size = "nano"
selected_version = "11"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}

model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}.pt"
model = YOLO(MODELS_DIRECTORY + model_name)

### 3.2 Training Configuration
Define hyperparameters (epochs, batch size, image size)

In [ ]:
import yaml

data_yaml = dict(
    train=os.path.join('../../', TRAINING_DATASET_DIRECTORY),
    val=os.path.join('../../', VALIDATION_DATASET_DIRECTORY),
    test=os.path.join('../../', TEST_DATASET_DIRECTORY),
    nc=2,
    names=['drone', 'other']
)

data_config_path = os.path.join(DATASET_ROOT_DIR, 'data.yaml')

with open(data_config_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=True)

%cat "$data_config_path"

In [ ]:
train_config = {
    'data': data_config_path,
    'epochs': 2,
    'imgsz': 640,
    'batch': 16,
    'lr0': 0.001,
    'optimizer': 'AdamW',
    'project': 'computer-vision',
    'name': selected_version + "-" + selected_size,
    'device': [0]
}

train_config

**Parameter breakdown:**

- **train**: Executes the YOLOv11x training pipeline.
- **model**=yolov11x.pt: Uses pre-trained YOLOv11x weights as initialization.
- **data**=/content/data.yaml: Specifies the dataset configuration file.
- **imgsz**=640: Sets input resolution to enhance small-object detection.
- **lr0**=0.001: Sets the learing rate for each training step.
- **epochs**=32: Defines the number of training cycles over the dataset.
- **batch**=16: Sets the batch size for each training step.
- **device**=0: Allocates GPU device 0 for training.
- **optimizer**=AdamW: As default, AdamW optimization algorithm utilized.

### 3.3 Run Training

In [ ]:
results = model.train(**train_config)

### 3.4 Result of training

In [ ]:
# print(results.save_dir)

In [ ]:
result_dir = results.save_dir
display(IPyImage(filename=os.path.join(result_dir, "results.png")))

In [ ]:
results_paths = []

for root, _, files in os.walk(result_dir):
    for f in files:
        if f.lower().endswith((".jpg")):
            results_paths.append(os.path.join(root, f))
results_paths = sorted(results_paths)

In [ ]:
%matplotlib inline

for path in results_paths:
    image = np.array(Image.open(path))
    plt.figure(figsize=(20, 10))
    plt.imshow(image)
    plt.axis("off")
    plt.show()


### 3.5 Evaluate Model


In [ ]:
metrics = model.val()
metrics

### 3.6 Compare with previous trains

In [ ]:
saved_train_path = "./saved_trains"

# List all folders in the saved_train_path
folders_results = [dir for dir in os.listdir(saved_train_path) if os.path.isdir(os.path.join(saved_train_path, dir))]

# Convert folder names to datetime objects for sorting
folders_results_sorted = sorted(
    folders_results,
    key=lambda x: datetime.strptime(x, "%d-%m-%Y_%H:%M:%S"),
    reverse=True  # latest first
)

# Take the last 2 trainings by date
last_two_folders = folders_results_sorted[:2]

# Dictionary to store the last row of each folder
results_data = {}

for folder in last_two_folders:
    path = os.path.join(saved_train_path, folder)
    results_csv = os.path.join(path, "results.csv")

    if os.path.exists(results_csv):
        df = pd.read_csv(results_csv)
        # Get the last row as a dictionary
        last_row = df.iloc[-1].to_dict()

        # Compute a custom YOLO train score
        score = (
                last_row['metrics/mAP50-95(B)'] * 0.7 +
                last_row['metrics/recall(B)'] * 0.2 -
                last_row['val/box_loss'] * 0.1
        )

        # Add the score to the dictionary
        last_row['train_score'] = score

        results_data[folder] = last_row

# Convert the dictionary to a pandas DataFrame
summary_df = pd.DataFrame.from_dict(results_data, orient='index')
summary_df.reset_index(inplace=True)
summary_df.rename(columns={'index': 'folder'}, inplace=True)

# Sort by train_score if desired
best_train = summary_df.sort_values(by='train_score', ascending=False)

# 4. Evaluation / Inference

Purpose: Evaluate results qualitatively and quantitatively.


### 4.0 Load model

In [ ]:
# best_path = os.path.join(result_dir, "weights/best.pt")
best_path = os.path.join("./computer-vision/nano11", "weights/best.pt")
custom_model = YOLO(best_path)

### 4.1 Run Predictions (IMAGES)

In [ ]:
preds = custom_model.predict(VALIDATION_DATASET_DIRECTORY, save=True,
                             project=os.path.join("./computer-vision/nano11", "runs/"))
predictions_output_dir = preds[0].save_dir

### 4.2 Result of Prediction

In [ ]:
results = custom_model.predict(VALIDATION_DATASET_DIRECTORY, conf=0.2)

### 4.3 Visualize Predictions

In [ ]:
predictions_paths = []

for root, _, files in os.walk(predictions_output_dir):
    for f in files:
        if f.lower().endswith((".jpg", ".png", ".jpeg")):
            predictions_paths.append(os.path.join(root, f))

In [ ]:
cols = 4
max_images_preview = -1
rows = math.ceil(len(predictions_paths) / cols)

plt.figure(figsize=(cols * 3, rows * 2))  # dynamic size
for i, path in enumerate(predictions_paths):
    if max_images_preview != -1 and max_images_preview < len(predictions_paths):
        break
    img = Image.open(path)
    plt.subplot(rows, cols, i + 1)
    plt.imshow(img)
    plt.axis("off")

plt.tight_layout()
plt.show()

### 4.4 Run Predictions (VIDEOS)

In [ ]:
result = custom_model.track(
    source="video1.mp4",
    conf=0.3,
    iou=0.5,
    show=False,
    imgsz=640,
    save=True,
    project=predictions_output_dir,
    name="tracking_output",
    exist_ok=True  # overwrite if folder exists
)

# 📊 5. Analysis & Improvements
Purpose: Interpret and iterate.

### 5.1 Metrics Review

In [ ]:
metrics.box.map, metrics.box.map50, metrics.box.map75

### 5.2 Error Analysis

In [ ]:
metrics.confusion_matrix.plot()

### 5.3 Compare with previous trains

# ✅ 6. Export & Deployment

### 6.1 Save results

In [ ]:
# source = os.path.join(RUNS_DIRECTORY, "train")
# saved_success_file_name = ".saved.success"
#
# check_for_file = os.path.isfile(os.path.join(source, saved_success_file_name))
#
# if check_for_file:
#     print("Training already saved. If you want to save it again, delete the .saved.success file.")
# else:
#     now = datetime.now().strftime('%d-%m-%Y_%H:%M:%S')
#     destination = "./saved_trains/" + now
#
#     shutil.copytree(source, destination, dirs_exist_ok=True)
#
#     train_config_path = os.path.join(destination, 'train_config.json')
#     with open(train_config_path, 'w') as outfile:
#         yaml.dump(json.dumps(train_config), outfile, default_flow_style=True)
#
#     saved_file = os.path.join(source, '.saved.success')
#
#     with open(saved_file, "a") as f:
#         f.write(now)
#     print("Training saved successfully.")